# EchoSentinel offline-style inference test in Google Colab

This notebook runs the project's existing `backend/audio_processor.py` and `backend/model.py` against one uploaded audio sample. It does not change the production model, preprocessing, label mapping, or 0.5 decision threshold. The reported softmax values are uncalibrated screening scores, not real-world accuracy or proof of authenticity.

## Before you run

1. Open this notebook in Google Colab and select **Runtime → Change runtime type**. A GPU is optional; CPU inference is supported but may be slow.
2. The model cache on your Mac is **not** available in Colab. Colab checks its own Hugging Face cache first. If the exact pinned checkpoint is absent, the notebook estimates its size and asks before fetching the checkpoint.
3. Only the two required backend source modules are fetched from the supplied GitHub repository, over HTTPS, after showing their sizes and asking. No repository credentials are used. If the repository is private or unavailable anonymously, the notebook stops; do not paste tokens into it.
4. Missing Python packages are resolved first with a pip dry run. The notebook displays planned wheel sizes, checks the cumulative 4 GB artifact budget and Colab free space, then asks before downloading/installing. It installs wheels from a local wheelhouse with network access disabled during installation.
5. Model weights are not fetched unless the exact checkpoint is missing from Colab's cache and you approve that separate download. No dataset is downloaded.
6. The sample uploader accepts one audio file only (up to the project's 25 MB limit). Uploading sends that sample to your Colab runtime. Do not upload sensitive or private recordings unless you are authorized to do so.

The notebook reserves 64 MiB from the 4 GB artifact budget for package/model metadata and transfer overhead. Its download ledger is stored under `/content`. If you resume in a new runtime after earlier downloads, enter the prior artifact byte total when prompted; the notebook cannot infer transfers from a lost runtime.


In [ ]:
import importlib.metadata as metadata
import json
import os
import shutil
import sys
from pathlib import Path

WORKDIR = Path('/content/echosentinel_colab')
SOURCE_DIR = WORKDIR / 'backend'
WHEEL_DIR = WORKDIR / 'wheelhouse'
WORKDIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
WHEEL_DIR.mkdir(parents=True, exist_ok=True)
LEDGER_PATH = Path('/content/echosentinel_download_ledger.json')
DOWNLOAD_BUDGET_BYTES = 4_000_000_000
METADATA_RESERVE_BYTES = 64 * 1024 * 1024
ARTIFACT_LIMIT_BYTES = DOWNLOAD_BUDGET_BYTES - METADATA_RESERVE_BYTES

def validate_ledger(value):
    if not isinstance(value, dict) or set(('artifact_bytes', 'artifacts')) - value.keys():
        raise RuntimeError('Download ledger is malformed; refusing to continue.')
    total = value['artifact_bytes']
    items = value['artifacts']
    if isinstance(total, bool) or not isinstance(total, int) or total < 0 or total > ARTIFACT_LIMIT_BYTES:
        raise RuntimeError('Download ledger has an invalid cumulative byte count; refusing to continue.')
    if not isinstance(items, list):
        raise RuntimeError('Download ledger artifact list is malformed; refusing to continue.')
    item_total = 0
    for item in items:
        if (not isinstance(item, dict) or not isinstance(item.get('name'), str)
                or isinstance(item.get('bytes'), bool) or not isinstance(item.get('bytes'), int)
                or item['bytes'] < 0):
            raise RuntimeError('Download ledger contains an invalid artifact entry; refusing to continue.')
        item_total += item['bytes']
    if item_total != total:
        raise RuntimeError('Download ledger entries do not match its total; refusing to continue.')
    return value

def save_ledger():
    temporary_path = LEDGER_PATH.with_suffix('.json.tmp')
    temporary_path.write_text(json.dumps(ledger, indent=2))
    os.replace(temporary_path, LEDGER_PATH)

if LEDGER_PATH.exists():
    try:
        ledger = validate_ledger(json.loads(LEDGER_PATH.read_text()))
    except (OSError, json.JSONDecodeError) as exc:
        raise RuntimeError('Cannot read the existing download ledger safely; refusing to continue.') from exc
else:
    prior = input('Prior artifact bytes already downloaded for this notebook workflow? Enter 0 only for a first run: ').strip()
    if not prior.isdigit() or int(prior) > ARTIFACT_LIMIT_BYTES:
        raise RuntimeError('Enter a valid prior byte total within the budget; stopping.')
    prior_bytes = int(prior)
    ledger = {'artifact_bytes': prior_bytes, 'artifacts': ([{'name': 'prior workflow downloads', 'bytes': prior_bytes}] if prior_bytes else [])}
    save_ledger()

def record_artifact(name, size):
    global ledger
    size = int(size)
    if size < 0 or ledger['artifact_bytes'] + size > ARTIFACT_LIMIT_BYTES:
        raise RuntimeError('Download budget would be exceeded; stopping before this artifact.')
    ledger['artifact_bytes'] += size
    ledger['artifacts'].append({'name': str(name), 'bytes': size})
    save_ledger()

def available_bytes():
    return shutil.disk_usage('/content').free

def ensure_storage(required_bytes, reserve_bytes=1024**3):
    free = available_bytes()
    print(f'Colab free disk: {free / 1024**3:.2f} GiB; required estimate plus reserve: {required_bytes / 1024**3:.2f} GiB + {reserve_bytes / 1024**3:.2f} GiB')
    if free < required_bytes + reserve_bytes:
        raise RuntimeError('Insufficient Colab disk space; no download or installation will start.')

print('Python:', sys.version.split()[0], sys.executable)
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch not importable:', type(exc).__name__)
print('Colab runtime status: active notebook kernel; GPU status is reported above.')
print(f'Colab free disk: {available_bytes() / 1024**3:.2f} GiB')
artifacts_used_gb = ledger['artifact_bytes'] / 1e9
print(f"Artifact budget: {DOWNLOAD_BUDGET_BYTES / 1e9:.1f} GB; artifacts used: {artifacts_used_gb:.3f} GB; metadata/transfer reserve: {METADATA_RESERVE_BYTES / 1024**2:.0f} MiB")


## 1. Fetch only the project's inference modules

The project is not cloned wholesale. This cell obtains only `backend/model.py` and `backend/audio_processor.py` from the public repository at the inspected immutable commit `3dfdb3eda1a9e85df1b9579fc27948b8c2482315`. It uses HTTPS certificate verification, requires a known `Content-Length`, displays the exact combined size, checks the budget and disk reserve, and waits for explicit approval. It verifies each downloaded file against its Git blob ID recorded at that commit and prints SHA-256 hashes so the exact source used can be recorded. If GitHub denies anonymous access or no safe size estimate is available, it stops without requesting credentials.


In [ ]:
import hashlib
from urllib.parse import urlparse
import requests

REPO_RAW = 'https://raw.githubusercontent.com/krishporwal011/EchoSentinel/3dfdb3eda1a9e85df1b9579fc27948b8c2482315'
SOURCE_FILES = ('backend/model.py', 'backend/audio_processor.py')
SOURCE_GIT_BLOBS = {
    'backend/model.py': 'f25b329df88995a3cea9c145923fbcef54e4676c',
    'backend/audio_processor.py': 'c8f82b4ea0a845894992c815742c0a0626f77771',
}
source_plan = []
for rel in SOURCE_FILES:
    url = f'{REPO_RAW}/{rel}'
    head = requests.head(url, allow_redirects=True, timeout=30, headers={'Accept-Encoding':'identity'}, verify=True)
    if head.status_code != 200 or not head.url.startswith('https://'):
        raise RuntimeError(f'Cannot safely access public source file {rel} (HTTP {head.status_code}). No credentials were used.')
    length = head.headers.get('Content-Length')
    if not length or not length.isdigit():
        raise RuntimeError(f'No reliable size estimate for {rel}; refusing to download.')
    source_plan.append((rel, url, int(length)))
source_bytes = sum(item[2] for item in source_plan)
print('Public source files and estimated sizes:')
for rel, _, size in source_plan:
    print(f'  {rel}: {size:,} bytes ({size / 1024:.1f} KiB)')
print(f'Combined source size: {source_bytes:,} bytes')
projected_source_total_gb = (ledger['artifact_bytes'] + source_bytes) / 1e9
print(f"Cumulative artifacts after source: {projected_source_total_gb:.3f} GB of {ARTIFACT_LIMIT_BYTES / 1e9:.3f} GB available for artifacts")
ensure_storage(source_bytes + 16 * 1024**2)
if ledger['artifact_bytes'] + source_bytes > ARTIFACT_LIMIT_BYTES:
    raise RuntimeError('Source files would exceed the download budget.')
approval = input('Download these two public source files over HTTPS? Type FETCH SOURCE to approve: ').strip()
if approval != 'FETCH SOURCE':
    raise RuntimeError('Source download declined; stopping.')

for rel, url, expected in source_plan:
    record_artifact(rel, expected)  # reserve/count before transfer so an interrupted request cannot evade the cumulative cap
    response = requests.get(url, stream=True, timeout=60, headers={'Accept-Encoding':'identity'}, verify=True)
    response.raise_for_status()
    actual_header = response.headers.get('Content-Length')
    if not response.url.startswith('https://'):
        response.close()
        raise RuntimeError('Source request redirected away from HTTPS; refusing it.')
    if actual_header and actual_header.isdigit() and int(actual_header) != expected:
        response.close()
        raise RuntimeError(f'Source size changed after preflight for {rel}; refusing to save it.')
    output = WORKDIR / rel
    output.parent.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256()
    git_blob_digest = hashlib.sha1(f'blob {expected}\0'.encode('ascii'))
    received = 0
    with output.open('wb') as handle:
        for chunk in response.iter_content(chunk_size=64 * 1024):
            if not chunk:
                continue
            received += len(chunk)
            if received > expected:
                raise RuntimeError('Source response exceeded its approved size/budget; stopped while streaming.')
            digest.update(chunk)
            git_blob_digest.update(chunk)
            handle.write(chunk)
    if received != expected:
        raise RuntimeError(f'Incomplete source response for {rel}: expected {expected}, received {received}.')
    if git_blob_digest.hexdigest() != SOURCE_GIT_BLOBS[rel]:
        raise RuntimeError(f'GitHub blob hash mismatch for {rel} at the pinned commit; refusing to import it.')
    print(f'Fetched {rel}: {received:,} bytes; SHA-256 {digest.hexdigest()}; pinned Git blob verified')

sys.path.insert(0, str(SOURCE_DIR))
print('Project modules are available from:', SOURCE_DIR)


## 2. Preflight and approve only missing/incompatible direct dependencies

The inference path needs the dependencies used by the existing modules: PyTorch, Transformers, NumPy, Librosa, SoundFile, and Certifi. FastAPI, Uvicorn, Pydantic, and the web frontend are not needed for this single-file notebook test. Versions below match `backend/requirements.txt` where pinned; PyTorch retains the project's `>=2.4.0` constraint. The resolver report includes required transitive wheels.

A pip dry run may fetch small package-index metadata, but it does not fetch wheel archives or install packages. This notebook reserves 64 MiB for this metadata and protocol overhead. If a wheel size cannot be measured safely, it aborts before installation.


In [ ]:
import json
import subprocess
import requests
from urllib.parse import urlparse, unquote

PACKAGE_SPECS = {
    'torch': 'torch>=2.4.0',
    'transformers': 'transformers==5.17.0',
    'librosa': 'librosa==1.0.0',
    'soundfile': 'soundfile==0.14.0',
    'numpy': 'numpy==2.5.3',
    'certifi': 'certifi==2026.7.22',
    'packaging': 'packaging>=24.0',
}

# Resolve only distributions that are missing or do not satisfy the project's declared version.
from packaging.requirements import Requirement
from packaging.version import Version
install_specs = []
for dist, spec_text in PACKAGE_SPECS.items():
    requirement = Requirement(spec_text)
    try:
        installed = metadata.version(dist)
    except metadata.PackageNotFoundError:
        installed = None
    compatible = installed is not None and (not requirement.specifier or Version(installed) in requirement.specifier)
    installed_text = installed if installed else 'missing'
    requirement_note = '' if compatible else f' -> requires {spec_text}'
    print(f"{dist}: {installed_text}{requirement_note}")
    if not compatible:
        install_specs.append(spec_text)

if not install_specs:
    wheel_plan = []
    print('All required packages satisfy the project requirements; no package install is needed.')
else:
    report_path = WORKDIR / 'pip-dry-run-report.json'
    command = [sys.executable, '-m', 'pip', 'install', '--dry-run', '--only-binary=:all:', '--report', str(report_path), *install_specs]
    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode != 0:
        print((result.stderr or result.stdout)[-3000:])
        raise RuntimeError('Dependency resolution failed; no package archives were installed or downloaded.')
    report = json.loads(report_path.read_text())
    wheel_plan = []
    for item in report.get('install', []):
        info = item['download_info']
        url = info['url']
        if not url.startswith('https://'):
            raise RuntimeError('Resolver returned a non-HTTPS artifact URL; refusing it.')
        head = requests.head(url, allow_redirects=True, timeout=30, headers={'Accept-Encoding':'identity'}, verify=True)
        length = head.headers.get('Content-Length')
        if head.status_code >= 400 or not head.url.startswith('https://') or not length or not length.isdigit():
            raise RuntimeError(f"Could not safely estimate wheel size for {item['metadata']['name']}; stopping before download.")
        filename = unquote(urlparse(url).path.rsplit('/', 1)[-1])
        if not filename.endswith('.whl'):
            raise RuntimeError(f"No binary wheel is available for {item['metadata']['name']}; refusing a source build.")
        wheel_plan.append({'name': item['metadata']['name'], 'version': item['metadata']['version'], 'url': url, 'filename': filename, 'bytes': int(length)})

planned_wheel_bytes = sum(item['bytes'] for item in wheel_plan)
# Budget conservatively for installed files plus wheelhouse temporary storage.
install_storage_estimate = planned_wheel_bytes * 3
model_expected_bytes = 378_351_004  # pinned checkpoint snapshot: weights + two JSON files, verified from the local cache
print('Planned package artifacts:')
for item in wheel_plan:
    print(f"  {item['name']}=={item['version']}: {item['bytes']:,} bytes")
print(f'Package download estimate: {planned_wheel_bytes:,} bytes ({planned_wheel_bytes / 1e9:.3f} GB)')
print(f'Conservative package storage estimate (download + installed/temp allowance): {install_storage_estimate / 1024**3:.2f} GiB')
print(f'Known pinned model snapshot estimate if not cached in Colab: {model_expected_bytes:,} bytes ({model_expected_bytes / 1024**2:.1f} MiB)')
print(f"Cumulative artifact estimate (source + wheels + model if needed): {(ledger['artifact_bytes'] + planned_wheel_bytes + model_expected_bytes) / 1e9:.3f} GB; artifact cap {ARTIFACT_LIMIT_BYTES / 1e9:.3f} GB")
if ledger['artifact_bytes'] + planned_wheel_bytes + model_expected_bytes > ARTIFACT_LIMIT_BYTES:
    raise RuntimeError('Worst-case package + model artifacts exceed the guarded 4 GB budget; stopping before package downloads.')
ensure_storage(install_storage_estimate + model_expected_bytes + 32 * 1024**2)

if wheel_plan:
    approval = input('Download and install the listed project dependencies? Type INSTALL DEPENDENCIES to approve: ').strip()
    if approval != 'INSTALL DEPENDENCIES':
        raise RuntimeError('Dependency installation declined; no wheel archives were fetched.')
    wheel_paths = []
    for item in wheel_plan:
        record_artifact(item['filename'], item['bytes'])  # reserve/count before transfer; conservative if interrupted
        response = requests.get(item['url'], stream=True, timeout=120, headers={'Accept-Encoding':'identity'}, verify=True)
        response.raise_for_status()
        length = response.headers.get('Content-Length')
        if not response.url.startswith('https://') or not length or not length.isdigit() or int(length) != item['bytes']:
            response.close()
            raise RuntimeError(f"Wheel size changed or is unknown for {item['name']}; refusing to download it.")
        path = WHEEL_DIR / item['filename']
        received = 0
        with path.open('wb') as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                received += len(chunk)
                if received > item['bytes']:
                    raise RuntimeError('Wheel response exceeded its approved size/budget; stopped while streaming.')
                handle.write(chunk)
        if received != item['bytes']:
            raise RuntimeError(f"Incomplete wheel for {item['name']}: expected {item['bytes']}, received {received}.")
        wheel_paths.append(str(path))
    install = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--no-cache-dir', *wheel_paths], text=True, capture_output=True)
    print(install.stdout[-3000:])
    if install.returncode != 0:
        print(install.stderr[-3000:])
        raise RuntimeError('Offline installation from the approved wheelhouse failed.')
    print('Approved wheels installed without pip network access.')
else:
    print('No installation approval required.')

print('Artifact ledger:', f"{ledger['artifact_bytes']:,} / {ARTIFACT_LIMIT_BYTES:,} bytes")


## 3. Reuse or explicitly fetch the pinned model checkpoint

The notebook uses the model ID `HyperMoon/wav2vec2-base-960h-finetuned-deepfake` at immutable revision `c73de548282dd35b6dec0844b924e3a931f3abfc`, matching the checkpoint already cached on the Mac. That Mac cache is not mounted in Colab. The known snapshot is 378,351,004 bytes (about 361 MiB). Colab first checks its own Hugging Face cache. If missing, the cell queries metadata for only the three required files, verifies the expected aggregate size, checks the remaining cumulative budget and disk reserve, then asks for approval.


In [ ]:
from huggingface_hub import HfApi, snapshot_download, try_to_load_from_cache

MODEL_ID = 'HyperMoon/wav2vec2-base-960h-finetuned-deepfake'
MODEL_REVISION = 'c73de548282dd35b6dec0844b924e3a931f3abfc'
MODEL_FILES = ('config.json', 'preprocessor_config.json', 'pytorch_model.bin')
EXPECTED_MODEL_BYTES = 378_351_004

cached = {}
for filename in MODEL_FILES:
    try:
        cached[filename] = try_to_load_from_cache(MODEL_ID, filename, revision=MODEL_REVISION)
    except Exception:
        cached[filename] = None
cache_complete = all(path and Path(path).is_file() for path in cached.values())
if cache_complete:
    MODEL_PATH = Path(cached[MODEL_FILES[0]]).parent
    cached_model_size = sum(Path(cached[name]).stat().st_size for name in MODEL_FILES)
    if cached_model_size != EXPECTED_MODEL_BYTES:
        raise RuntimeError(f'Cached pinned checkpoint size mismatch: expected {EXPECTED_MODEL_BYTES}, found {cached_model_size}; refusing inference.')
    print('Exact pinned checkpoint found in this Colab Hugging Face cache and its aggregate size matches; no model download is needed.')
else:
    print('The exact pinned checkpoint is not fully present in the Colab cache.')
    info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION, files_metadata=True)
    remote_sizes = {item.rfilename: item.size for item in info.siblings if item.rfilename in MODEL_FILES}
    if set(remote_sizes) != set(MODEL_FILES) or any(size is None for size in remote_sizes.values()):
        raise RuntimeError('Could not verify the sizes of all required pinned model files; refusing to download.')
    total_model_size = sum(int(remote_sizes[name]) for name in MODEL_FILES)
    if total_model_size != EXPECTED_MODEL_BYTES:
        raise RuntimeError(f'Pinned model size changed: expected {EXPECTED_MODEL_BYTES}, metadata reports {total_model_size}; refusing to download.')
    missing_model_size = sum(int(remote_sizes[name]) for name in MODEL_FILES if not (cached[name] and Path(cached[name]).is_file()))
    print('Model files and metadata sizes:')
    for name in MODEL_FILES:
        print(f'  {name}: {int(remote_sizes[name]):,} bytes' + (' (cached)' if cached[name] and Path(cached[name]).is_file() else ' (to fetch)'))
    print(f'Model bytes to fetch: {missing_model_size:,} ({missing_model_size / 1024**2:.1f} MiB)')
    if ledger['artifact_bytes'] + missing_model_size > ARTIFACT_LIMIT_BYTES:
        raise RuntimeError('Model download would exceed the cumulative 4 GB artifact budget.')
    ensure_storage(missing_model_size + 32 * 1024**2)
    approval = input('Fetch only the missing files of this pinned model checkpoint? Type DOWNLOAD MODEL to approve: ').strip()
    if approval != 'DOWNLOAD MODEL':
        raise RuntimeError('Model download declined; inference cannot proceed without a complete cached checkpoint.')
    before_present = {name: bool(cached[name] and Path(cached[name]).is_file()) for name in MODEL_FILES}
    record_artifact('pinned model checkpoint', missing_model_size)  # reserve/count before download; conservative if interrupted
    model_dir = snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION, allow_patterns=list(MODEL_FILES))
    MODEL_PATH = Path(model_dir)
    actual_model_size = sum((MODEL_PATH / name).stat().st_size for name in MODEL_FILES)
    if actual_model_size != EXPECTED_MODEL_BYTES:
        raise RuntimeError(f'Downloaded snapshot size mismatch: expected {EXPECTED_MODEL_BYTES}, found {actual_model_size}.')
    downloaded_model_bytes = sum((MODEL_PATH / name).stat().st_size for name in MODEL_FILES if not before_present[name])
    if downloaded_model_bytes > missing_model_size:
        raise RuntimeError('Model artifact bytes exceeded the approved estimate.')
    print(f'Pinned model checkpoint is ready; up to {missing_model_size:,} bytes were reserved in the ledger before fetching.')

for filename in MODEL_FILES:
    if not (MODEL_PATH / filename).is_file():
        raise RuntimeError(f'Required model file is missing: {filename}')
print('Model ID:', MODEL_ID)
print('Pinned revision:', MODEL_REVISION)
print('Local model path:', MODEL_PATH)
print('Final artifact ledger:', f"{ledger['artifact_bytes']:,} / {ARTIFACT_LIMIT_BYTES:,} bytes (64 MiB reserved for metadata/overhead)")


## 4. Upload one small audio sample and run the project pipeline

Supported extensions match the backend: WAV, FLAC, OGG, MP3, M4A, and AAC. The existing preprocessor enforces a 25 MiB file-size limit, a 0.5–60 second duration, non-silence checks, mono downmixing, 16 kHz resampling, and peak normalization. It may require the codec support bundled in the Colab runtime.

The notebook uses this repository revision's `decode_and_preprocess_audio(..., return_meta=True)` and `VoiceCloneDetector.predict` directly. That model wrapper returns `prediction`, `confidence`, `model_spoof_score`, `risk_score`, and `risk_level`; its `confidence` is the uncalibrated selected-class softmax score, not real-world accuracy. The wrapper resolves bonafide/spoof indices from the checkpoint config and uses its existing 0.5 model decision threshold. It does not return a separate unrounded human-class score, so the notebook does not invent one.


In [ ]:
from google.colab import files
import json
import torch

MAX_UPLOAD_BYTES = 25 * 1024 * 1024
SUPPORTED_EXTENSIONS = {'.wav', '.flac', '.ogg', '.mp3', '.m4a', '.aac'}
consent = input('This uploads one audio sample to your Colab runtime. Type I AUTHORIZE AUDIO UPLOAD to continue: ').strip()
if consent != 'I AUTHORIZE AUDIO UPLOAD':
    raise RuntimeError('Audio upload declined; no audio was selected.')
ensure_storage(MAX_UPLOAD_BYTES + 32 * 1024**2)
selected = files.upload()
if len(selected) != 1:
    raise RuntimeError('Upload exactly one audio file.')
filename, audio_bytes = next(iter(selected.items()))
extension = Path(filename).suffix.lower()
if extension not in SUPPORTED_EXTENSIONS:
    raise ValueError(f'Unsupported extension {extension!r}; choose WAV, FLAC, OGG, MP3, M4A, or AAC.')
if not audio_bytes or len(audio_bytes) > MAX_UPLOAD_BYTES:
    raise ValueError('Audio must be non-empty and no larger than 25 MiB.')
ensure_storage(len(audio_bytes) + 32 * 1024**2)

from audio_processor import decode_and_preprocess_audio
from model import VoiceCloneDetector

waveform, source_sample_rate = decode_and_preprocess_audio(audio_bytes, return_meta=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
detector = VoiceCloneDetector(model_id=str(MODEL_PATH), device=device)
detector.load_model()
result = detector.predict(waveform, sample_rate=16000)
spoof_score = float(result['model_spoof_score'])
if not 0.0 <= spoof_score <= 1.0:
    raise RuntimeError('Model returned an out-of-range spoof score; refusing to display a successful prediction.')
response = {
    'success': True,
    'prediction': result['prediction'],
    'confidence': result['confidence'],
    'confidence_note': 'Uncalibrated selected-class softmax score; not real-world accuracy or proof.',
    'model_spoof_score': spoof_score,
    'risk_score': int(result['risk_score']),
    'risk_level': result['risk_level'],
    'processing_time_ms': float(result['processing_time_ms']),
    'audio': {
        'duration_seconds': round(float(len(waveform) / 16000), 3),
        'preprocessor_reported_sample_rate': int(source_sample_rate),
        'model_sample_rate': 16000,
    },
    'model': {
        'name': MODEL_ID,
        'revision': MODEL_REVISION,
        'architecture': detector.model.__class__.__name__,
        'inference_completed': True,
        'labels': detector.label_mapping,
    },
    'notice': 'These are uncalibrated model estimates, not definitive proof that a recording is genuine or synthetic.',
}
print(json.dumps(response, indent=2))


## Troubleshooting and safe interpretation

- **Repository 404:** the notebook fetches only public source files and does not accept GitHub credentials. Confirm the repository is publicly readable at the supplied URL before retrying.
- **Dependency plan too large / insufficient disk:** stop. Do not remove preinstalled Colab packages to make room. Use a fresh Colab runtime with more available disk or wait until a compatible runtime is available.
- **Checkpoint fetch declined or unavailable:** no inference can run until the pinned checkpoint is present in Colab's cache. The cache on your Mac is not shared automatically.
- **Unsupported/corrupt/silent audio:** choose one non-empty supported recording between 0.5 and 60 seconds and under 25 MiB.
- **Prediction interpretation:** the wrapper returns the selected class’ uncalibrated softmax score in `confidence`, and the spoof-class softmax score in `model_spoof_score`. Neither is calibrated real-world accuracy; a high AI score is not proof. Audio quality and model score are separate concepts.
- **Runtime or memory failures:** retry on a Colab GPU runtime if available. A GPU is optional and no model settings are changed by the notebook.

This is a one-sample inference smoke test, not an evaluation of model accuracy. It does not download any dataset, start the FastAPI server, or change the local project.
